# 04 — Single-config backtest

Run :class:`bowaka_lab.sim.portfolio_engine.BowakaPortfolioBacktester` against
the candidates emitted by notebook **03**. Produces:

- ``trades.parquet`` — one row per simulated trade.
- ``summary.json`` — aggregate stats (trade_count, win_rate, mean_pnl_pct,
  total_pnl, exits_by_reason).
- ``config.json`` — full backtest config snapshot for reproducibility.

This notebook does NOT generate the final weekly report — that's notebook 11.
It DOES print the funnel from notebook 03 for sanity.


In [ ]:
# Notebook bootstrap cell. Keep this in every bowaka_lab notebook.
from pathlib import Path
import sys

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "research_notebooks").exists():
    repo_root = repo_root.parent

bowaka_project = repo_root / "research_notebooks" / "bowaka_lab"
src_path = bowaka_project / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import bowaka_lab
from bowaka_lab.utils.env import load_project_dotenv

_loaded_env = load_project_dotenv()
print(f"bowaka_lab {bowaka_lab.__version__}")
print(
    f"bowaka_lab bootstrap: .env loaded from {_loaded_env}"
    if _loaded_env
    else "bowaka_lab bootstrap: no .env found (env vars must be set in shell)"
)


## Configuration

In [ ]:
import os

RUN_ID         = "bt_iex_default"
DATA_ROOT      = os.environ.get(
    "BOWAKA_DATA_ROOT",
    "research_notebooks/bowaka_lab/db_tools/bowaka_data",
)
ARTIFACTS_ROOT = "research_notebooks/bowaka_lab/artifacts"
# Swap to ``configs/bowaka_exact_current_strategy.yml`` for the source-strategy
# paper-mode profile.
CONFIG_PATH    = "research_notebooks/bowaka_lab/configs/bowaka_research_variant.yml"
REBUILD        = False


## Paths + backtest config

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

from bowaka_lab.config import (
    assert_exact_mode_invariants,
    compute_config_hash,
    load_config_file,
)
from bowaka_lab.data.calendar import USEquityCalendar
from bowaka_lab.data.parquet_io import MinuteBarLoader, candidates_dict_to_source
from bowaka_lab.metrics.trade_metrics import per_trade_metrics, summary_stats
from bowaka_lab.metrics.diagnostics import exit_reason_distribution
from bowaka_lab.sim.portfolio_engine import BowakaPortfolioBacktester
from bowaka_lab.utils import (
    ArtifactPaths,
    artifact_exists,
    load_json,
    load_parquet,
    save_json,
    save_parquet,
)


data_root      = Path(DATA_ROOT)      if Path(DATA_ROOT).is_absolute()      else (repo_root / DATA_ROOT).resolve()
artifacts_root = Path(ARTIFACTS_ROOT) if Path(ARTIFACTS_ROOT).is_absolute() else (repo_root / ARTIFACTS_ROOT).resolve()
config_path    = Path(CONFIG_PATH)    if Path(CONFIG_PATH).is_absolute()    else (repo_root / CONFIG_PATH).resolve()

cfg = load_config_file(config_path)
assert_exact_mode_invariants(cfg)
config_hash = compute_config_hash(cfg)

MINUTE_ROOT = data_root / "parquet/bars/vendor=alpaca" / f"feed={cfg.data.feed}" / "timeframe=1m/adjustment=raw"

paths = ArtifactPaths.for_run(RUN_ID, artifacts_root)
paths.ensure_dir()
assert paths.candidates.exists(), (
    f"candidates artifact missing: {paths.candidates}\n"
    "Run notebook 03_prefilter_replay first."
)

cal = USEquityCalendar(cfg.calendar.exchange)
print(f"config:        {config_path}")
print(f"config_hash:   {config_hash}")
print(f"fidelity_mode: {cfg.project.fidelity_mode}")
print(f"artifacts:     {paths.root}")
print(f"minute root:   {MINUTE_ROOT}")
print(f"entry:         {cfg.entry.default_rule}, slip={cfg.entry.slippage_bps}bps")
print(f"exits:         stop={cfg.exits.stop_pct} target={cfg.exits.target_pct} hold={cfg.exits.max_hold_days}")


## Load candidates from notebook 03

In [ ]:
candidates_df = load_parquet(paths.candidates)
print(f"candidates loaded: {candidates_df.shape[0]:,} rows")

# Reshape into the dict the backtester expects: signal_date -> per-day DataFrame.
candidate_frames = {
    sd: g.reset_index(drop=True)
    for sd, g in candidates_df.groupby("signal_date", sort=False)
}
candidate_source = candidates_dict_to_source(candidate_frames)
minute_bars_for  = MinuteBarLoader(MINUTE_ROOT)
print(f"signal dates with candidates: {len(candidate_frames):,}")


## Run backtest

In [ ]:
trades_df = None
summary = None

if not REBUILD and artifact_exists(paths, "trades") and artifact_exists(paths, "summary"):
    print("Fast path: trades.parquet + summary.json already exist; loading.")
    trades_df = load_parquet(paths.trades)
    summary = load_json(paths.summary)
else:
    runner = BowakaPortfolioBacktester(
        cfg,
        candidate_source=candidate_source,
        minute_bars_for=minute_bars_for,
        calendar=cal,
    )
    result = runner.run()
    trades_df = result.trades_df()

    save_parquet(paths.trades, trades_df)
    save_json(paths.config, cfg.model_dump(mode="json"))

    if trades_df.empty:
        summary = {"trade_count": 0, "win_rate": 0.0, "mean_pnl_pct": 0.0,
                   "total_pnl": 0.0, "exits_by_reason": {}}
    else:
        scored = per_trade_metrics(trades_df, stop_pct=cfg.exits.stop_pct)
        stats = summary_stats(scored)
        exits = (scored["exit_reason"].value_counts().to_dict()
                 if "exit_reason" in scored.columns else {})
        summary = {**stats, "exits_by_reason": exits}
    save_json(paths.summary, summary)

print(f"trades:  {trades_df.shape[0]:,}")
print(f"wrote {paths.trades}")
print(f"wrote {paths.summary}")
print(f"wrote {paths.config}")


## Trade summary + exit reasons

In [ ]:
import json

print(json.dumps({k: v for k, v in summary.items() if k != "exits_by_reason"}, indent=2))
print()
print("Exits by reason:")
for reason, count in (summary.get("exits_by_reason") or {}).items():
    print(f"  {reason}: {count}")

if not trades_df.empty:
    try:
        from IPython.display import display
        display(trades_df.head(10))
    except Exception:
        print(trades_df.head(10).to_string(index=False))


## Equity curve

In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

if not trades_df.empty and plt is not None and "pnl" in trades_df.columns:
    daily = trades_df.groupby("trade_date")["pnl"].sum().reset_index()
    daily["cumulative_pnl"] = daily["pnl"].cumsum()
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(daily["trade_date"].astype(str), daily["cumulative_pnl"], marker="o", linewidth=1)
    ax.set_title(f"Cumulative PnL ($) — {RUN_ID}")
    ax.set_xlabel("trade_date")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    plt.show()
else:
    print("(no trades to plot or matplotlib unavailable)")


## Funnel sanity (read from notebook 03 artifact)

In [ ]:
if artifact_exists(paths, "funnel"):
    funnel = load_json(paths.funnel)
    totals = {k: v for k, v in funnel.items() if k != "per_session"}
    print("Prefilter funnel (from notebook 03):")
    for k, v in totals.items():
        print(f"  {k}: {int(v):,}")
    assert int(funnel.get("candidates", 0)) > 0, (
        "Funnel reports zero candidates — re-run notebook 03 with REBUILD=True."
    )
else:
    print("funnel.json missing — run notebook 03 first.")


## Next

- Run **05/06/07/08** for counterfactuals, exits, signal-fade, and liquidity
  analysis.
- Run **11_weekly_research_report.ipynb** to aggregate everything into the
  final Markdown + JSON report.